# rMATS显著 event → 8 classes → 检查污染基因 → QC排除 → 提取每个脑区×8类的基因 → 看基因数量是否足够富集。

# 一，rmats富集分析

# Cell 1：参数

In [ ]:
# %% Cell 1：参数

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
from tqdm.auto import tqdm

GENE_DIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260705_Fig3/anno_h5ads"
)

RMATS_DIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260705_Fig3/rmats_works/rmats_results"
)

OUT_DIR = Path(
    "/histor/zhao/projects/lupeng/zhaolab/ZJJ/"
    "Fig4_AS_enrichment/6class_direction_read10"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES = ["G21-4", "G21-7", "G21-1", "G21-8", "G21-2", "G21-5"]

COMPARISONS = ["control_vs_AD", "control_vs_old"]

REGIONS = [
    "HIP", "CA1", "CA3", "DG",
    "CTX_1", "CTX_2", "CTX_3",
    "TH", "HY", "FT", "STR", "VS",
]

AS_TYPES = ["SE", "MXE", "A3SS", "A5SS", "RI"]

READ_THRESH = 10
DPSI_THRESH = 0.05
FDR_THRESH = 0.05
ENRICH_PADJ = 0.05
MIN_GENES = 10

GROUPS = [
    "AD only +",
    "AD only -",
    "Shared +/+",
    "Shared -/-",
    "Aged only +",
    "Aged only -",
]

# Cell 2：QC 排除基因

In [ ]:
# %% Cell 2：QC排除基因

EXCLUDE_GENES = [
    "Cox8b", "Actc1", "Acta1", "Myl7", "Tnni3",
    "Ckmt2", "Ckm", "Csrp3", "Hrc", "Tcap",
    "Klhl40", "Sln", "Pgam2",
    "Krt10", "Kap", "Aldob", "Spink1", "Htra4",
    "Miox", "Napsa", "Gkn3", "Car6", "Crisp3",
    "Pip", "Defb11", "Mucl2", "Scgb1b27", "Lct", "Tsacc",
    "Klk1", "Klk1b1", "Klk1b3", "Klk1b4", "Klk1b5",
    "Klk1b8", "Klk1b9", "Klk1b11", "Klk1b16",
    "Klk1b21", "Klk1b22", "Klk1b26", "Klk1b27",
    "Scgb2b26", "Scgb2b27", "Cd3d", "Cd3g", "Cox7a1",
    "Fxyd2", "Gm15543", "Ly6d", "S100g", "Trbc1",
    "Wfdc12", "Fabp4", "Fbp1",
    "Gh", "Guca2b", "Nppb", "Nppa", "Tnnc1",
    "Wfdc15b", "Dmkn", "Krt15",
    "Trbc2", "Wfdc18", "Cox6a2", "Krt8", "Krt18",
]

ALL_EXCLUDE_GENES = set(EXCLUDE_GENES)

# Cell 3：六个基因矩阵取交集，建立 background

In [ ]:
# %% Cell 3：构建background

gene_sets = []
annotations = []

for sample in tqdm(SAMPLES, desc="Reading genes"):
    ad = sc.read_h5ad(GENE_DIR / f"{sample}_regions.h5ad", backed="r")

    gene_sets.append(set(ad.var_names.astype(str)))

    annotations.append(
        pd.DataFrame({
            "gene_id": ad.var_names.astype(str),
            "geneSymbol": ad.var["Symbol"].astype("string").values,
        })
    )

    print(sample, ad.n_vars)
    ad.file.close()

# 六个样本共同基因
common_gene_ids = set.intersection(*gene_sets)

# gene_id → geneSymbol
annotation = pd.concat(annotations, ignore_index=True)
annotation = annotation.dropna(subset=["geneSymbol"]).drop_duplicates("gene_id")

BACKGROUND_GENES = (
    annotation.loc[
        annotation["gene_id"].isin(common_gene_ids),
        "geneSymbol"
    ]
    .dropna()
    .astype(str)
    .unique()
)

# 删除QC基因
BACKGROUND_GENES = sorted(
    set(BACKGROUND_GENES) - ALL_EXCLUDE_GENES
)

print("Common gene IDs:", len(common_gene_ids))
print("Background genes after QC:", len(BACKGROUND_GENES))

# Cell 4：重新读取 rMATS，使用 READ=10

In [ ]:
# %% Cell 4：读取rMATS

EVENT_COLS = {
    "SE": ["chr", "strand", "exonStart_0base", "exonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
    "MXE": ["chr", "strand", "1stExonStart_0base", "1stExonEnd", "2ndExonStart_0base", "2ndExonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
    "A3SS": ["chr", "strand", "longExonStart_0base", "longExonEnd", "shortES", "shortEE", "flankingES", "flankingEE"],
    "A5SS": ["chr", "strand", "longExonStart_0base", "longExonEnd", "shortES", "shortEE", "flankingES", "flankingEE"],
    "RI": ["chr", "strand", "riExonStart_0base", "riExonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
}


def counts(x):
    return x.astype(str).str.split(",", expand=True).apply(pd.to_numeric, errors="coerce").fillna(0)


def filter_events(df):
    total1 = counts(df["IJC_SAMPLE_1"]) + counts(df["SJC_SAMPLE_1"])
    total2 = counts(df["IJC_SAMPLE_2"]) + counts(df["SJC_SAMPLE_2"])

    dpsi = pd.to_numeric(df["IncLevelDifference"], errors="coerce")
    fdr = pd.to_numeric(df["FDR"], errors="coerce")

    mask = (
        (total1 >= READ_THRESH).all(axis=1) &
        (total2 >= READ_THRESH).all(axis=1) &
        (dpsi.abs() > DPSI_THRESH) &
        (fdr <= FDR_THRESH)
    )

    return df.loc[mask].copy()

# Cell 5：显著AS events

In [ ]:
# %% Cell 5：显著AS events

tables = []

for comparison in COMPARISONS:
    for region in REGIONS:
        for ast in AS_TYPES:

            file = RMATS_DIR / comparison / region / f"{ast}.MATS.JC.txt"
            df = filter_events(pd.read_csv(file, sep="\t"))

            df["Comparison"] = comparison
            df["Region"] = region
            df["AS_type"] = ast

            df["EventID"] = ast + "|" + df[
                ["GeneID"] + EVENT_COLS[ast]
            ].astype(str).agg("|".join, axis=1)

            tables.append(
                df[[
                    "Comparison", "Region", "EventID",
                    "AS_type", "GeneID", "geneSymbol",
                    "IncLevelDifference"
                ]]
            )

events = pd.concat(tables, ignore_index=True)

print("Significant events:", len(events))

# Cell 6：重新分成 8 classes

In [ ]:
# %% Cell 6：8 classes

wide = events.pivot_table(
    index=["Region", "EventID", "AS_type", "GeneID", "geneSymbol"],
    columns="Comparison",
    values="IncLevelDifference",
    aggfunc="first",
).reset_index()

ad = wide["control_vs_AD"]
aged = wide["control_vs_old"]

wide["AS_class"] = pd.Series(pd.NA, index=wide.index, dtype="string")

wide.loc[ad.notna() & aged.isna() & (ad > 0), "AS_class"] = "AD only +"
wide.loc[ad.notna() & aged.isna() & (ad < 0), "AS_class"] = "AD only -"

wide.loc[ad.notna() & aged.notna() & (ad > 0) & (aged > 0), "AS_class"] = "Shared +/+"
wide.loc[ad.notna() & aged.notna() & (ad < 0) & (aged < 0), "AS_class"] = "Shared -/-"
wide.loc[ad.notna() & aged.notna() & (ad > 0) & (aged < 0), "AS_class"] = "Shared +/-"
wide.loc[ad.notna() & aged.notna() & (ad < 0) & (aged > 0), "AS_class"] = "Shared -/+"

wide.loc[ad.isna() & aged.notna() & (aged > 0), "AS_class"] = "Aged only +"
wide.loc[ad.isna() & aged.notna() & (aged < 0), "AS_class"] = "Aged only -"

display(wide["AS_class"].value_counts())

# Cell 7：QC + 六类 foreground genes

In [ ]:
# %% Cell 7：提取富集基因

wide_qc = wide[
    ~wide["geneSymbol"].isin(ALL_EXCLUDE_GENES)
].copy()

genes = (
    wide_qc[
        wide_qc["AS_class"].isin(GROUPS)
    ][["Region", "AS_class", "GeneID", "geneSymbol"]]
    .dropna(subset=["geneSymbol"])
    .drop_duplicates()
)

# 只保留background中的基因
genes_not_background = genes[
    ~genes["geneSymbol"].isin(BACKGROUND_GENES)
].copy()

genes = genes[
    genes["geneSymbol"].isin(BACKGROUND_GENES)
].copy()

print("Foreground genes outside background:", genes_not_background["geneSymbol"].nunique())

gene_counts = (
    genes
    .groupby(["Region", "AS_class"])["geneSymbol"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=REGIONS, columns=GROUPS, fill_value=0)
)

display(gene_counts)

# Cell 8：富集数据库

In [ ]:
# %% Cell 8：富集数据库

GENE_SETS = {
    "GO_BP": "GO_Biological_Process_2026",
    "GO_MF": "GO_Molecular_Function_2026",
    "KEGG": "KEGG_2019_Mouse",
}

libraries = {
    db: gp.get_library(name=lib, organism="Mouse")
    for db, lib in GENE_SETS.items()
}

# Cell 9：正式富集

In [ ]:
# %% Cell 9：通路富集

all_results = []

for region in tqdm(REGIONS, desc="Regions"):
    for group in GROUPS:

        gene_list = sorted(
            genes.loc[
                (genes["Region"] == region) &
                (genes["AS_class"] == group),
                "geneSymbol"
            ].unique()
        )

        if len(gene_list) < MIN_GENES:
            continue

        for db, gene_set_dict in libraries.items():

            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=gene_set_dict,
                background=BACKGROUND_GENES,
                outdir=None,
                no_plot=True,
                cutoff=1.0,
                verbose=False,
            )

            res = enr.results.copy()

            if res.empty:
                continue

            res.insert(0, "Region", region)
            res.insert(1, "AS_class", group)
            res.insert(2, "database", db)
            res.insert(3, "n_input_genes", len(gene_list))
            res.insert(4, "n_background_genes", len(BACKGROUND_GENES))

            all_results.append(res)

enrich_all = pd.concat(all_results, ignore_index=True)

enrich_sig = enrich_all[
    enrich_all["Adjusted P-value"] < ENRICH_PADJ
].copy()

print("Background genes:", len(BACKGROUND_GENES))
print("All enrichment terms:", len(enrich_all))
print("Significant terms:", len(enrich_sig))

In [ ]:
# %% Cell 10：显著通路数量

enrich_count = (
    enrich_sig
    .groupby(["Region", "AS_class", "database"])
    .size()
    .reset_index(name="n_terms")
)

display(enrich_count)

# Cell 11：全部保存

In [ ]:
# %% Cell 11：保存

genes.to_csv(
    OUT_DIR / "AS_6class_genes_for_enrichment.csv",
    index=False
)

genes_not_background.to_csv(
    OUT_DIR / "AS_genes_outside_background.csv",
    index=False
)

gene_counts.to_csv(
    OUT_DIR / "AS_6class_gene_counts.csv"
)

enrich_all.to_csv(
    OUT_DIR / "AS_6class_enrichment_all_results.csv",
    index=False
)

enrich_sig.to_csv(
    OUT_DIR / "AS_6class_enrichment_sig_padj0.05.csv",
    index=False
)

enrich_count.to_csv(
    OUT_DIR / "AS_6class_enrichment_term_counts.csv",
    index=False
)

print("Saved:", OUT_DIR)

# 二、Cell 1：合并方向并提取基因

In [ ]:
# %% Cell 1：合并方向

GROUPS_3 = ["AD only", "Shared", "Aged only"]

group_map = {
    "AD only +": "AD only",
    "AD only -": "AD only",
    "Shared +/+": "Shared",
    "Shared -/-": "Shared",
    "Shared +/-": "Shared",
    "Shared -/+": "Shared",
    "Aged only +": "Aged only",
    "Aged only -": "Aged only",
}

genes_3class = wide_qc[
    ["Region", "AS_class", "GeneID", "geneSymbol"]
].copy()

genes_3class["AS_group"] = genes_3class["AS_class"].map(group_map)

genes_3class = (
    genes_3class
    .dropna(subset=["AS_group", "geneSymbol"])
    .drop_duplicates(["Region", "AS_group", "geneSymbol"])
)

# 只保留background中的基因
genes_3class = genes_3class[
    genes_3class["geneSymbol"].isin(BACKGROUND_GENES)
].copy()

display(genes_3class.head())

In [ ]:
# %% Cell 2：基因数量

gene_counts_3class = (
    genes_3class
    .groupby(["Region", "AS_group"])["geneSymbol"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=REGIONS, columns=GROUPS_3, fill_value=0)
)

display(gene_counts_3class)

## Cell 3：三类通路富集

In [ ]:
# %% Cell 3：三类通路富集

all_results_3class = []

for region in tqdm(REGIONS, desc="Regions"):
    for group in GROUPS_3:

        gene_list = sorted(
            genes_3class.loc[
                (genes_3class["Region"] == region) &
                (genes_3class["AS_group"] == group),
                "geneSymbol"
            ].unique()
        )

        if len(gene_list) < 10:
            continue

        for db, gene_set_dict in libraries.items():

            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=gene_set_dict,
                background=BACKGROUND_GENES,
                outdir=None,
                no_plot=True,
                cutoff=1.0,
                verbose=False,
            )

            res = enr.results.copy()

            if res.empty:
                continue

            res.insert(0, "Region", region)
            res.insert(1, "AS_group", group)
            res.insert(2, "database", db)
            res.insert(3, "n_input_genes", len(gene_list))
            res.insert(4, "n_background_genes", len(BACKGROUND_GENES))

            all_results_3class.append(res)

enrich_all_3class = pd.concat(
    all_results_3class,
    ignore_index=True
)

enrich_sig_3class = enrich_all_3class[
    enrich_all_3class["Adjusted P-value"] < 0.05
].copy()

print("Background genes:", len(BACKGROUND_GENES))
print("All enrichment terms:", len(enrich_all_3class))
print("Significant terms:", len(enrich_sig_3class))

In [ ]:
# %% Cell 4：显著通路数量

enrich_count_3class = (
    enrich_sig_3class
    .groupby(["Region", "AS_group", "database"])
    .size()
    .reset_index(name="n_terms")
)

display(enrich_count_3class)

In [ ]:
# %% Cell 5：保存

OUT_DIR_3 = Path(
    "/histor/zhao/projects/lupeng/zhaolab/ZJJ/"
    "Fig4_AS_enrichment/3class_no_direction_read10"
)

OUT_DIR_3.mkdir(parents=True, exist_ok=True)

genes_3class.to_csv(
    OUT_DIR_3 / "AS_3class_genes_for_enrichment.csv",
    index=False
)

gene_counts_3class.to_csv(
    OUT_DIR_3 / "AS_3class_gene_counts.csv"
)

enrich_all_3class.to_csv(
    OUT_DIR_3 / "AS_3class_enrichment_all_results.csv",
    index=False
)

enrich_sig_3class.to_csv(
    OUT_DIR_3 / "AS_3class_enrichment_sig_padj0.05.csv",
    index=False
)

enrich_count_3class.to_csv(
    OUT_DIR_3 / "AS_3class_enrichment_term_counts.csv",
    index=False
)

print("Saved:", OUT_DIR_3)

# 三、富集通路的基因绘制筛选

In [ ]:
# %% Cell 1：参数

from pathlib import Path
import numpy as np
import pandas as pd

RMATS_DIR = Path("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/rmats_works/rmats_results")
ENRICH_FILE = Path("/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_AS_enrichment/6class_direction_read10/AS_6class_enrichment_sig_padj0.05.csv")
OUT_DIR = Path("/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_AS_trackplot_candidates")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COMPARISONS = ["control_vs_AD", "control_vs_old"]
REGIONS = ["HIP", "CA1", "CA3", "DG", "CTX_1", "CTX_2", "CTX_3", "TH", "HY", "FT", "STR", "VS"]
AS_TYPES = ["SE", "MXE", "A3SS", "A5SS", "RI"]

READ_THRESH = 10
DPSI_THRESH = 0.05
FDR_THRESH = 0.05

GROUPS = ["AD only +", "AD only -", "Shared +/+", "Shared -/-", "Aged only +", "Aged only -"]

EVENT_COLS = {
    "SE": ["chr", "strand", "exonStart_0base", "exonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
    "MXE": ["chr", "strand", "1stExonStart_0base", "1stExonEnd", "2ndExonStart_0base", "2ndExonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
    "A3SS": ["chr", "strand", "longExonStart_0base", "longExonEnd", "shortES", "shortEE", "flankingES", "flankingEE"],
    "A5SS": ["chr", "strand", "longExonStart_0base", "longExonEnd", "shortES", "shortEE", "flankingES", "flankingEE"],
    "RI": ["chr", "strand", "riExonStart_0base", "riExonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
}

def split_num(x):
    return x.astype(str).str.split(",", expand=True).apply(pd.to_numeric, errors="coerce")

In [ ]:
# %% Cell 2：读取并过滤rMATS

tables = []

for comparison in COMPARISONS:
    for region in REGIONS:
        for ast in AS_TYPES:
            file = RMATS_DIR / comparison / region / f"{ast}.MATS.JC.txt"
            df = pd.read_csv(file, sep="\t")

            ijc1 = split_num(df["IJC_SAMPLE_1"])
            sjc1 = split_num(df["SJC_SAMPLE_1"])
            ijc2 = split_num(df["IJC_SAMPLE_2"])
            sjc2 = split_num(df["SJC_SAMPLE_2"])
            total1 = ijc1 + sjc1
            total2 = ijc2 + sjc2

            psi1 = split_num(df["IncLevel1"])
            psi2 = split_num(df["IncLevel2"])

            fdr = pd.to_numeric(df["FDR"], errors="coerce")
            dpsi = -pd.to_numeric(df["IncLevelDifference"], errors="coerce")

            mask = (total1 >= READ_THRESH).all(axis=1) & (total2 >= READ_THRESH).all(axis=1) & (dpsi.abs() > DPSI_THRESH) & (fdr <= FDR_THRESH)
            df = df.loc[mask].copy()

            df["dPSI"] = dpsi.loc[mask]
            df["FDR"] = fdr.loc[mask]
            df["min_junction"] = pd.concat([total1, total2], axis=1).min(axis=1).loc[mask]
            df["mean_junction"] = pd.concat([total1, total2], axis=1).mean(axis=1).loc[mask]
            df["psi_range"] = pd.concat([psi1.max(axis=1) - psi1.min(axis=1), psi2.max(axis=1) - psi2.min(axis=1)], axis=1).max(axis=1).loc[mask]

            df["Comparison"] = comparison
            df["Region"] = region
            df["AS_type"] = ast
            df["EventID"] = ast + "|" + df[["GeneID"] + EVENT_COLS[ast]].astype(str).agg("|".join, axis=1)

            tables.append(df[[
                "Comparison", "Region", "EventID", "AS_type", "GeneID", "geneSymbol",
                "dPSI", "FDR", "min_junction", "mean_junction", "psi_range",
                "IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2",
                "IncLevel1", "IncLevel2"
            ]])

events_detail = pd.concat(tables, ignore_index=True)

print("Filtered comparison-level events:", len(events_detail))
display(events_detail.head())

In [ ]:
# %% Cell 3：8类事件

INDEX_COLS = ["Region", "EventID", "AS_type", "GeneID", "geneSymbol"]
VALUE_COLS = ["dPSI", "FDR", "min_junction", "mean_junction", "psi_range",
              "IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2",
              "IncLevel1", "IncLevel2"]

events_wide = events_detail.pivot_table(index=INDEX_COLS, columns="Comparison", values=VALUE_COLS, aggfunc="first").reset_index()
events_wide.columns = ["_".join([str(x) for x in col if str(x)]) if isinstance(col, tuple) else col for col in events_wide.columns]

ad = events_wide["dPSI_control_vs_AD"]
aged = events_wide["dPSI_control_vs_old"]

events_wide["AS_class"] = pd.NA

events_wide.loc[ad.notna() & aged.isna() & (ad > 0), "AS_class"] = "AD only +"
events_wide.loc[ad.notna() & aged.isna() & (ad < 0), "AS_class"] = "AD only -"
events_wide.loc[ad.notna() & aged.notna() & (ad > 0) & (aged > 0), "AS_class"] = "Shared +/+"
events_wide.loc[ad.notna() & aged.notna() & (ad < 0) & (aged < 0), "AS_class"] = "Shared -/-"
events_wide.loc[ad.notna() & aged.notna() & (ad > 0) & (aged < 0), "AS_class"] = "Shared +/-"
events_wide.loc[ad.notna() & aged.notna() & (ad < 0) & (aged > 0), "AS_class"] = "Shared -/+"
events_wide.loc[ad.isna() & aged.notna() & (aged > 0), "AS_class"] = "Aged only +"
events_wide.loc[ad.isna() & aged.notna() & (aged < 0), "AS_class"] = "Aged only -"

display(events_wide["AS_class"].value_counts())

In [ ]:
# %% Cell 4：画图筛选指标

events_wide["plot_abs_dPSI"] = np.nan
events_wide["plot_min_junction"] = np.nan
events_wide["plot_psi_range"] = np.nan
events_wide["plot_FDR"] = np.nan

m = events_wide["AS_class"].str.startswith("AD only", na=False)
events_wide.loc[m, "plot_abs_dPSI"] = events_wide.loc[m, "dPSI_control_vs_AD"].abs()
events_wide.loc[m, "plot_min_junction"] = events_wide.loc[m, "min_junction_control_vs_AD"]
events_wide.loc[m, "plot_psi_range"] = events_wide.loc[m, "psi_range_control_vs_AD"]
events_wide.loc[m, "plot_FDR"] = events_wide.loc[m, "FDR_control_vs_AD"]

m = events_wide["AS_class"].str.startswith("Aged only", na=False)
events_wide.loc[m, "plot_abs_dPSI"] = events_wide.loc[m, "dPSI_control_vs_old"].abs()
events_wide.loc[m, "plot_min_junction"] = events_wide.loc[m, "min_junction_control_vs_old"]
events_wide.loc[m, "plot_psi_range"] = events_wide.loc[m, "psi_range_control_vs_old"]
events_wide.loc[m, "plot_FDR"] = events_wide.loc[m, "FDR_control_vs_old"]

m = events_wide["AS_class"].str.startswith("Shared", na=False)
events_wide.loc[m, "plot_abs_dPSI"] = events_wide.loc[m, ["dPSI_control_vs_AD", "dPSI_control_vs_old"]].abs().min(axis=1)
events_wide.loc[m, "plot_min_junction"] = events_wide.loc[m, ["min_junction_control_vs_AD", "min_junction_control_vs_old"]].min(axis=1)
events_wide.loc[m, "plot_psi_range"] = events_wide.loc[m, ["psi_range_control_vs_AD", "psi_range_control_vs_old"]].max(axis=1)
events_wide.loc[m, "plot_FDR"] = events_wide.loc[m, ["FDR_control_vs_AD", "FDR_control_vs_old"]].max(axis=1)

events_6class = events_wide[events_wide["AS_class"].isin(GROUPS)].copy()

events_wide.to_csv(OUT_DIR / "AS_filtered_events_8class_detailed.csv", index=False)
events_6class.to_csv(OUT_DIR / "AS_filtered_events_6class_detailed.csv", index=False)

print("6-class events:", len(events_6class))

In [ ]:
# %% Cell 5：筛选Trackplot候选

TOP_PATHWAYS = 3
PLOT_DPSI = 0.15
PLOT_READ = 20
MAX_PSI_RANGE = 0.20

# 读取显著富集结果
enrich = pd.read_csv(ENRICH_FILE)

# 旧富集结果方向标签翻转
CLASS_FLIP = {
    "AD only +": "AD only -",
    "AD only -": "AD only +",
    "Shared +/+": "Shared -/-",
    "Shared -/-": "Shared +/+",
    "Aged only +": "Aged only -",
    "Aged only -": "Aged only +",
}
enrich["AS_class"] = enrich["AS_class"].replace(CLASS_FLIP)

# 每个脑区×类别取Top3显著通路
top_terms = (
    enrich.sort_values("Adjusted P-value")
    .groupby(["Region", "AS_class"], group_keys=False)
    .head(TOP_PATHWAYS)
    .copy()
)

# 提取通路基因
pathway_genes = top_terms[["Region", "AS_class", "Genes"]].copy()
pathway_genes["gene_key"] = pathway_genes["Genes"].str.replace(",", ";", regex=False).str.split(";")
pathway_genes = pathway_genes.explode("gene_key")
pathway_genes["gene_key"] = pathway_genes["gene_key"].str.strip().str.upper()
pathway_genes = pathway_genes[["Region", "AS_class", "gene_key"]].drop_duplicates()

# 匹配event
events_match = events_6class.copy()
events_match["gene_key"] = events_match["geneSymbol"].astype(str).str.upper()

candidates = events_match.merge(
    pathway_genes,
    on=["Region", "AS_class", "gene_key"],
    how="inner"
)

# 筛选适合画图的event
candidates = candidates[
    (candidates["plot_abs_dPSI"] >= PLOT_DPSI) &
    (candidates["plot_min_junction"] >= PLOT_READ) &
    (candidates["plot_psi_range"] <= MAX_PSI_RANGE)
].copy()

print("Candidates:")
display(candidates.groupby("AS_class").size().reindex(GROUPS, fill_value=0))

In [ ]:
# %% Cell 6：每类Top20基因

N_PER_CLASS = 20

# 排序
candidates = candidates.sort_values(
    ["plot_abs_dPSI", "plot_min_junction", "plot_psi_range", "plot_FDR"],
    ascending=[False, False, True, True]
)

# 每个基因只保留最好的一个event
top_genes = (
    candidates
    .drop_duplicates(["AS_class", "geneSymbol"])
    .groupby("AS_class", group_keys=False)
    .head(N_PER_CLASS)
    .copy()
)

top_genes["AS_class"] = pd.Categorical(top_genes["AS_class"], categories=GROUPS, ordered=True)
top_genes = top_genes.sort_values(["AS_class", "plot_abs_dPSI"], ascending=[True, False])

# 保存
top_genes.to_csv(OUT_DIR / "AS_trackplot_top_genes.csv", index=False)

display(top_genes[[
    "Region", "AS_class", "AS_type", "geneSymbol", "EventID",
    "plot_abs_dPSI", "plot_min_junction", "plot_psi_range", "plot_FDR"
]])

# 输出基因列表
for group in GROUPS:
    genes = top_genes.loc[top_genes["AS_class"] == group, "geneSymbol"].tolist()
    print(f"\n{group} ({len(genes)}):")
    print(", ".join(genes))

In [ ]:
# %% Cell 6：每类最多20个SE候选

N_PER_CLASS = 20

# 排序
candidates = candidates.sort_values(
    ["plot_abs_dPSI", "plot_min_junction", "plot_psi_range", "plot_FDR"],
    ascending=[False, False, True, True]
)

# 每类每个基因只保留最好的一个event
top_genes = (
    candidates
    .drop_duplicates(["AS_class", "geneSymbol"])
    .groupby("AS_class", group_keys=False)
    .head(N_PER_CLASS)
    .copy()
)

top_genes["AS_class"] = pd.Categorical(
    top_genes["AS_class"],
    categories=GROUPS,
    ordered=True
)

top_genes = top_genes.sort_values(
    ["AS_class", "plot_abs_dPSI"],
    ascending=[True, False]
)

# 保存
top_genes.to_csv(
    OUT_DIR / "SE_trackplot_top_candidates.csv",
    index=False
)

# 查看详细结果
display(top_genes[[
    "Region", "AS_class", "geneSymbol", "EventID",
    "plot_abs_dPSI", "plot_min_junction",
    "plot_psi_range", "plot_FDR"
]])

# 输出6类候选列表
for group in GROUPS:
    df = top_genes[top_genes["AS_class"] == group]

    print(f"\n===== {group} ({len(df)}) =====")

    for _, row in df.iterrows():
        print(
            f'{row["Region"]} | {row["geneSymbol"]} | '
            f'ΔPSI={row["plot_abs_dPSI"]:.3f} | '
            f'junction={row["plot_min_junction"]:.0f}'
        )

## SE版本

In [ ]:
# %% Cell 1：参数

from pathlib import Path
import numpy as np
import pandas as pd

RMATS_DIR = Path("/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/20260705_Fig3/rmats_works/rmats_results")
ENRICH_FILE = Path("/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_AS_enrichment/6class_direction_read10/AS_6class_enrichment_sig_padj0.05.csv")
OUT_DIR = Path("/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_AS_trackplot_candidates")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COMPARISONS = ["control_vs_AD", "control_vs_old"]
REGIONS = ["HIP", "CA1", "CA3", "DG", "CTX_1", "CTX_2", "CTX_3", "TH", "HY", "FT", "STR", "VS"]
AS_TYPES = ["SE",]

READ_THRESH = 10
DPSI_THRESH = 0.05
FDR_THRESH = 0.05

GROUPS = ["AD only +", "AD only -", "Shared +/+", "Shared -/-", "Aged only +", "Aged only -"]

EVENT_COLS = {
    "SE": ["chr", "strand", "exonStart_0base", "exonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
    "MXE": ["chr", "strand", "1stExonStart_0base", "1stExonEnd", "2ndExonStart_0base", "2ndExonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
    "A3SS": ["chr", "strand", "longExonStart_0base", "longExonEnd", "shortES", "shortEE", "flankingES", "flankingEE"],
    "A5SS": ["chr", "strand", "longExonStart_0base", "longExonEnd", "shortES", "shortEE", "flankingES", "flankingEE"],
    "RI": ["chr", "strand", "riExonStart_0base", "riExonEnd", "upstreamES", "upstreamEE", "downstreamES", "downstreamEE"],
}

def split_num(x):
    return x.astype(str).str.split(",", expand=True).apply(pd.to_numeric, errors="coerce")

In [ ]:
# %% Cell 2：读取并过滤rMATS

tables = []

for comparison in COMPARISONS:
    for region in REGIONS:
        for ast in AS_TYPES:
            file = RMATS_DIR / comparison / region / f"{ast}.MATS.JC.txt"
            df = pd.read_csv(file, sep="\t")

            ijc1 = split_num(df["IJC_SAMPLE_1"])
            sjc1 = split_num(df["SJC_SAMPLE_1"])
            ijc2 = split_num(df["IJC_SAMPLE_2"])
            sjc2 = split_num(df["SJC_SAMPLE_2"])
            total1 = ijc1 + sjc1
            total2 = ijc2 + sjc2

            psi1 = split_num(df["IncLevel1"])
            psi2 = split_num(df["IncLevel2"])

            fdr = pd.to_numeric(df["FDR"], errors="coerce")
            dpsi = -pd.to_numeric(df["IncLevelDifference"], errors="coerce")

            mask = (total1 >= READ_THRESH).all(axis=1) & (total2 >= READ_THRESH).all(axis=1) & (dpsi.abs() > DPSI_THRESH) & (fdr <= FDR_THRESH)
            df = df.loc[mask].copy()

            df["dPSI"] = dpsi.loc[mask]
            df["FDR"] = fdr.loc[mask]
            df["min_junction"] = pd.concat([total1, total2], axis=1).min(axis=1).loc[mask]
            df["mean_junction"] = pd.concat([total1, total2], axis=1).mean(axis=1).loc[mask]
            df["psi_range"] = pd.concat([psi1.max(axis=1) - psi1.min(axis=1), psi2.max(axis=1) - psi2.min(axis=1)], axis=1).max(axis=1).loc[mask]

            df["Comparison"] = comparison
            df["Region"] = region
            df["AS_type"] = ast
            df["EventID"] = ast + "|" + df[["GeneID"] + EVENT_COLS[ast]].astype(str).agg("|".join, axis=1)

            tables.append(df[[
                "Comparison", "Region", "EventID", "AS_type", "GeneID", "geneSymbol",
                "dPSI", "FDR", "min_junction", "mean_junction", "psi_range",
                "IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2",
                "IncLevel1", "IncLevel2"
            ]])

events_detail = pd.concat(tables, ignore_index=True)

print("Filtered comparison-level events:", len(events_detail))
display(events_detail.head())

In [ ]:
# %% Cell 3：8类事件

INDEX_COLS = ["Region", "EventID", "AS_type", "GeneID", "geneSymbol"]
VALUE_COLS = ["dPSI", "FDR", "min_junction", "mean_junction", "psi_range",
              "IJC_SAMPLE_1", "SJC_SAMPLE_1", "IJC_SAMPLE_2", "SJC_SAMPLE_2",
              "IncLevel1", "IncLevel2"]

events_wide = events_detail.pivot_table(index=INDEX_COLS, columns="Comparison", values=VALUE_COLS, aggfunc="first").reset_index()
events_wide.columns = ["_".join([str(x) for x in col if str(x)]) if isinstance(col, tuple) else col for col in events_wide.columns]

ad = events_wide["dPSI_control_vs_AD"]
aged = events_wide["dPSI_control_vs_old"]

events_wide["AS_class"] = pd.NA

events_wide.loc[ad.notna() & aged.isna() & (ad > 0), "AS_class"] = "AD only +"
events_wide.loc[ad.notna() & aged.isna() & (ad < 0), "AS_class"] = "AD only -"
events_wide.loc[ad.notna() & aged.notna() & (ad > 0) & (aged > 0), "AS_class"] = "Shared +/+"
events_wide.loc[ad.notna() & aged.notna() & (ad < 0) & (aged < 0), "AS_class"] = "Shared -/-"
events_wide.loc[ad.notna() & aged.notna() & (ad > 0) & (aged < 0), "AS_class"] = "Shared +/-"
events_wide.loc[ad.notna() & aged.notna() & (ad < 0) & (aged > 0), "AS_class"] = "Shared -/+"
events_wide.loc[ad.isna() & aged.notna() & (aged > 0), "AS_class"] = "Aged only +"
events_wide.loc[ad.isna() & aged.notna() & (aged < 0), "AS_class"] = "Aged only -"

display(events_wide["AS_class"].value_counts())

In [ ]:
# %% Cell 4：画图筛选指标

events_wide["plot_abs_dPSI"] = np.nan
events_wide["plot_min_junction"] = np.nan
events_wide["plot_psi_range"] = np.nan
events_wide["plot_FDR"] = np.nan

m = events_wide["AS_class"].str.startswith("AD only", na=False)
events_wide.loc[m, "plot_abs_dPSI"] = events_wide.loc[m, "dPSI_control_vs_AD"].abs()
events_wide.loc[m, "plot_min_junction"] = events_wide.loc[m, "min_junction_control_vs_AD"]
events_wide.loc[m, "plot_psi_range"] = events_wide.loc[m, "psi_range_control_vs_AD"]
events_wide.loc[m, "plot_FDR"] = events_wide.loc[m, "FDR_control_vs_AD"]

m = events_wide["AS_class"].str.startswith("Aged only", na=False)
events_wide.loc[m, "plot_abs_dPSI"] = events_wide.loc[m, "dPSI_control_vs_old"].abs()
events_wide.loc[m, "plot_min_junction"] = events_wide.loc[m, "min_junction_control_vs_old"]
events_wide.loc[m, "plot_psi_range"] = events_wide.loc[m, "psi_range_control_vs_old"]
events_wide.loc[m, "plot_FDR"] = events_wide.loc[m, "FDR_control_vs_old"]

m = events_wide["AS_class"].str.startswith("Shared", na=False)
events_wide.loc[m, "plot_abs_dPSI"] = events_wide.loc[m, ["dPSI_control_vs_AD", "dPSI_control_vs_old"]].abs().min(axis=1)
events_wide.loc[m, "plot_min_junction"] = events_wide.loc[m, ["min_junction_control_vs_AD", "min_junction_control_vs_old"]].min(axis=1)
events_wide.loc[m, "plot_psi_range"] = events_wide.loc[m, ["psi_range_control_vs_AD", "psi_range_control_vs_old"]].max(axis=1)
events_wide.loc[m, "plot_FDR"] = events_wide.loc[m, ["FDR_control_vs_AD", "FDR_control_vs_old"]].max(axis=1)

events_6class = events_wide[events_wide["AS_class"].isin(GROUPS)].copy()

events_wide.to_csv(OUT_DIR / "AS_filtered_events_8class_detailed.csv", index=False)
events_6class.to_csv(OUT_DIR / "AS_filtered_events_6class_detailed.csv", index=False)

print("6-class events:", len(events_6class))

In [ ]:
# %% Cell 5：筛选Trackplot候选

TOP_PATHWAYS = 3
PLOT_DPSI = 0.15
PLOT_READ = 20
MAX_PSI_RANGE = 0.20

# 读取显著富集结果
enrich = pd.read_csv(ENRICH_FILE)

# 旧富集结果方向标签翻转
CLASS_FLIP = {
    "AD only +": "AD only -",
    "AD only -": "AD only +",
    "Shared +/+": "Shared -/-",
    "Shared -/-": "Shared +/+",
    "Aged only +": "Aged only -",
    "Aged only -": "Aged only +",
}
enrich["AS_class"] = enrich["AS_class"].replace(CLASS_FLIP)

# 每个脑区×类别取Top3显著通路
top_terms = (
    enrich.sort_values("Adjusted P-value")
    .groupby(["Region", "AS_class"], group_keys=False)
    .head(TOP_PATHWAYS)
    .copy()
)

# 提取通路基因
pathway_genes = top_terms[["Region", "AS_class", "Genes"]].copy()
pathway_genes["gene_key"] = pathway_genes["Genes"].str.replace(",", ";", regex=False).str.split(";")
pathway_genes = pathway_genes.explode("gene_key")
pathway_genes["gene_key"] = pathway_genes["gene_key"].str.strip().str.upper()
pathway_genes = pathway_genes[["Region", "AS_class", "gene_key"]].drop_duplicates()

# 匹配event
events_match = events_6class.copy()
events_match["gene_key"] = events_match["geneSymbol"].astype(str).str.upper()

candidates = events_match.merge(
    pathway_genes,
    on=["Region", "AS_class", "gene_key"],
    how="inner"
)

# 筛选适合画图的event
candidates = candidates[
    (candidates["plot_abs_dPSI"] >= PLOT_DPSI) &
    (candidates["plot_min_junction"] >= PLOT_READ) &
    (candidates["plot_psi_range"] <= MAX_PSI_RANGE)
].copy()

print("Candidates:")
display(candidates.groupby("AS_class").size().reindex(GROUPS, fill_value=0))

In [ ]:
# %% Cell 6：每类最多20个SE候选

N_PER_CLASS = 20

# 排序
candidates = candidates.sort_values(
    ["plot_abs_dPSI", "plot_min_junction", "plot_psi_range", "plot_FDR"],
    ascending=[False, False, True, True]
)

# 每类每个基因只保留最好的一个event
top_genes = (
    candidates
    .drop_duplicates(["AS_class", "geneSymbol"])
    .groupby("AS_class", group_keys=False)
    .head(N_PER_CLASS)
    .copy()
)

top_genes["AS_class"] = pd.Categorical(
    top_genes["AS_class"],
    categories=GROUPS,
    ordered=True
)

top_genes = top_genes.sort_values(
    ["AS_class", "plot_abs_dPSI"],
    ascending=[True, False]
)

# 保存
top_genes.to_csv(
    OUT_DIR / "SE_trackplot_top_candidates.csv",
    index=False
)

# 查看详细结果
display(top_genes[[
    "Region", "AS_class", "geneSymbol", "EventID",
    "plot_abs_dPSI", "plot_min_junction",
    "plot_psi_range", "plot_FDR"
]])

# 输出6类候选列表
for group in GROUPS:
    df = top_genes[top_genes["AS_class"] == group]

    print(f"\n===== {group} ({len(df)}) =====")

    for _, row in df.iterrows():
        print(
            f'{row["Region"]} | {row["geneSymbol"]} | '
            f'ΔPSI={row["plot_abs_dPSI"]:.3f} | '
            f'junction={row["plot_min_junction"]:.0f}'
        )

# 000

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# 参数
MASTER_FILE = Path(
    "/histor/zhao/projects/lupeng/zhaolab/ZJJ/Fig4_Gene_isoform_AS/"
    "Gene_isoform_AS_master_table_with_usage_class.tsv.gz"
)
RMATS_DIR = Path(
    "/histor/zhao/pangkun/Zhaolab/Lupeng/Zhujunjie/"
    "20260705_Fig3/rmats_works/rmats_results"
)
OUT_DIR = Path(
    "/histor/zhao/projects/lupeng/zhaolab/ZJJ/"
    "Fig4_Gene_isoform_AS/red_point_rmats_lookup"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

GENE_FC = 0.5
READ_THRESH = 10
DPSI_THRESH = 0.05
FDR_THRESH = 0.05

COMPARISONS = {
    "AD_vs_Young": "control_vs_AD",
    "Aged_vs_Young": "control_vs_old",
}
KEY = ["comparison", "region", "gene_name"]

EVENT_COLS = {
    "SE": [
        "chr", "strand", "exonStart_0base", "exonEnd",
        "upstreamES", "upstreamEE", "downstreamES", "downstreamEE",
    ],
    "MXE": [
        "chr", "strand",
        "1stExonStart_0base", "1stExonEnd",
        "2ndExonStart_0base", "2ndExonEnd",
        "upstreamES", "upstreamEE", "downstreamES", "downstreamEE",
    ],
    "A3SS": [
        "chr", "strand", "longExonStart_0base", "longExonEnd",
        "shortES", "shortEE", "flankingES", "flankingEE",
    ],
    "A5SS": [
        "chr", "strand", "longExonStart_0base", "longExonEnd",
        "shortES", "shortEE", "flankingES", "flankingEE",
    ],
    "RI": [
        "chr", "strand", "riExonStart_0base", "riExonEnd",
        "upstreamES", "upstreamEE", "downstreamES", "downstreamEE",
    ],
}

def split_num(series):
    return series.astype(str).str.split(",", expand=True).apply(
        pd.to_numeric, errors="coerce"
    )


# 提取全部红色点，不提前限制脑区或基因
master = pd.read_csv(MASTER_FILE, sep="\t", low_memory=False)
as_changed = master["AS_changed"].astype(str).str.lower().eq("true")

red = master.loc[
    master["comparison"].isin(COMPARISONS)
    & master["log2FC"].abs().le(GENE_FC)
    & as_changed,
    KEY + ["gene_id", "log2FC", "padj", "max_abs_dPSI_sig"]
].copy()

assert not red.duplicated(KEY).any(), "master 中存在重复的 comparison–region–gene"

print("红色点数量：")
display(red.groupby("comparison").size().rename("n_region_gene"))


# 读取红色点对应的全部原始事件
tables = []

for (comparison, region), genes in red.groupby(["comparison", "region"]):
    n_events = 0

    for ast, columns in EVENT_COLS.items():
        file = RMATS_DIR / COMPARISONS[comparison] / region / f"{ast}.MATS.JC.txt"
        df = pd.read_csv(file, sep="\t", dtype={"GeneID": str}, low_memory=False)

        df["gene_name"] = (
            df["geneSymbol"].astype(str).str.strip()
            .str.replace('"', "", regex=False)
        )
        df = df[df["gene_name"].isin(genes["gene_name"])].copy()

        if df.empty:
            continue

        df["comparison"] = comparison
        df["region"] = region
        df["AS_type"] = ast
        df["source_file"] = str(file)

        # 保留原始坐标和原始 rMATS ID，增加可跨比较匹配的 EventID
        df["EventID"] = (
            ast + "|"
            + df[["GeneID"] + columns].astype(str).agg("|".join, axis=1)
        )

        # 原 Notebook：原始方向为 Young - Condition，取反后为 Condition - Young
        df["dPSI"] = -pd.to_numeric(df["IncLevelDifference"], errors="coerce")
        df["abs_dPSI"] = df["dPSI"].abs()
        df["FDR_num"] = pd.to_numeric(df["FDR"], errors="coerce")

        coverage_pass = pd.Series(True, index=df.index)

        for group in [1, 2]:
            ijc = split_num(df[f"IJC_SAMPLE_{group}"])
            sjc = split_num(df[f"SJC_SAMPLE_{group}"])
            psi = split_num(df[f"IncLevel{group}"])

            assert ijc.shape[1] == sjc.shape[1] == psi.shape[1], (
                f"样本列数量不一致：{file}"
            )

            total = ijc + sjc

            # 缺失 reads 不当作通过覆盖要求
            coverage_pass &= (
                ijc.notna().all(axis=1)
                & sjc.notna().all(axis=1)
                & total.ge(READ_THRESH).all(axis=1)
            )

            df[f"n_samples_{group}"] = ijc.shape[1]
            df[f"n_valid_PSI_{group}"] = psi.notna().sum(axis=1)

            df[f"min_IJC_{group}"] = ijc.min(axis=1)
            df[f"min_SJC_{group}"] = sjc.min(axis=1)
            df[f"min_total_junction_{group}"] = total.min(axis=1)
            df[f"mean_total_junction_{group}"] = total.mean(axis=1)

            df[f"mean_PSI_{group}"] = psi.mean(axis=1)
            df[f"sd_PSI_{group}"] = psi.std(axis=1)
            df[f"range_PSI_{group}"] = (
                psi.max(axis=1) - psi.min(axis=1)
            ).where(psi.notna().sum(axis=1).ge(2))

            # 原始逗号分隔列也保留，同时展开为单样本列
            for j in range(ijc.shape[1]):
                df[f"SAMPLE{group}_{j + 1}_IJC"] = ijc[j]
                df[f"SAMPLE{group}_{j + 1}_SJC"] = sjc[j]
                df[f"SAMPLE{group}_{j + 1}_PSI"] = psi[j]

        df["coverage_pass"] = coverage_pass
        df["min_total_junction"] = df[
            ["min_total_junction_1", "min_total_junction_2"]
        ].min(axis=1, skipna=False)
        df["max_within_group_PSI_range"] = df[
            ["range_PSI_1", "range_PSI_2"]
        ].max(axis=1, skipna=False)

        df["AS_sig"] = (
            df["coverage_pass"]
            & df["FDR_num"].le(FDR_THRESH)
            & df["abs_dPSI"].ge(DPSI_THRESH)
        )

        # 加入该 region–gene 的表达结果
        df = df.merge(genes, on=KEY, how="inner", validate="many_to_one")
        tables.append(df)
        n_events += len(df)

    print(f"{comparison} | {region}: {len(genes)} 个红色点，{n_events} 个原始事件")


# 汇总，不丢弃不显著或低覆盖事件
if not tables:
    raise ValueError("没有匹配到 rMATS 事件，请检查输入路径和基因名。")

events = pd.concat(tables, ignore_index=True)
events = events.sort_values(
    ["comparison", "region", "gene_name", "AS_sig", "abs_dPSI", "FDR_num"],
    ascending=[True, True, True, False, False, True],
)

front = KEY + [
    "gene_id", "log2FC", "padj", "max_abs_dPSI_sig",
    "AS_type", "EventID", "chr", "strand",
    "dPSI", "abs_dPSI", "FDR_num", "coverage_pass", "AS_sig",
    "min_total_junction", "max_within_group_PSI_range",
]
events = events[front + [c for c in events.columns if c not in front]]

# 只保存这一张表
OUT_FILE = OUT_DIR / "red_region_gene_rmats_all_events.tsv.gz"
events.to_csv(OUT_FILE, sep="\t", index=False, compression="gzip")

print("\n事件统计：")
display(
    events.groupby("comparison").agg(
        n_all_events=("EventID", "size"),
        n_significant_events=("AS_sig", "sum"),
    )
)

print("事件预览：")
display(events[front].head(20))

# 输出未匹配或未复现显著事件的红点，方便发现输入版本差异
matched = events.groupby(KEY).agg(
    n_raw_events=("EventID", "size"),
    n_sig_events=("AS_sig", "sum"),
).reset_index()
check = red.merge(matched, on=KEY, how="left")
unmatched = check[check["n_raw_events"].isna()]
no_sig = check[check["n_raw_events"].notna() & check["n_sig_events"].eq(0)]

print(f"未匹配到原始事件的红点：{len(unmatched)}")
print(f"匹配到事件但未通过当前 AS 阈值的红点：{len(no_sig)}")
if len(unmatched):
    display(unmatched)
if len(no_sig):
    display(no_sig)

print("\n请把这个文件发给我：")
print(OUT_FILE)